# Tahap 5 — Empirical Digital Twin / Canonical State Evaluation

Notebook ini mengevaluasi integritas representasi Canonical Twin State terhadap telemetry aktual. Fokusnya adalah schema conformity, source-to-state mapping, completeness, temporal integrity, data-quality behavior, preservasi nilai, dan determinisme—bukan forecasting. Implementasi komputasi tetap berada di `src/twin_state/`.

In [1]:
# Environment Check
from pathlib import Path
import importlib.metadata
import os
import platform
import sys

EXPECTED_DATASET_SHA256 = 'ca7831a188a191edbf82a673fac90dbb875b5095986ed07699c02530f2a02a0e'
REPOSITORY_URL = 'https://github.com/rehanalfarizu/new_jurnal.git'
IS_COLAB = 'google.colab' in sys.modules

def find_repository_root():
    candidates = [Path.cwd(), Path.cwd() / 'new_jurnal', Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'configs' / 'experiment.yaml').is_file() and (candidate / 'src').is_dir():
            return candidate.resolve()
    return None

REPO_ROOT = find_repository_root()
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Google Colab:', IS_COLAB)
print('Repository path:', REPO_ROOT or 'belum ditemukan')
for package in ['numpy', 'pandas', 'scikit-learn', 'matplotlib', 'PyYAML', 'ipykernel']:
    try:
        version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        version = 'belum terpasang'
    print(f'{package}: {version}')

Python: 3.11.9
Platform: macOS-15.7-x86_64-i386-64bit
Google Colab: False
Repository path: /Users/macbookpro/Documents/new_jurnal
numpy: 2.3.5
pandas: 2.3.3
scikit-learn: 1.8.0
matplotlib: 3.10.8
PyYAML: 6.0.3
ipykernel: 7.3.0


In [2]:
# Google Colab Setup
import subprocess

if IS_COLAB and REPO_ROOT is None:
    clone_target = Path('/content/new_jurnal')
    if not clone_target.exists():
        subprocess.run(['git', 'clone', REPOSITORY_URL, str(clone_target)], check=True)
    REPO_ROOT = clone_target.resolve()
elif REPO_ROOT is None:
    raise RuntimeError('Repository tidak ditemukan. Jalankan notebook dari root new_jurnal atau direktori notebooks/.')

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Working directory:', Path.cwd())

Working directory: /Users/macbookpro/Documents/new_jurnal


In [3]:
# Dependency Setup
import importlib.util

if IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
    print('Dependency dari requirements.txt telah dipasang.')
else:
    required_modules = {'pandas': 'pandas', 'PyYAML': 'yaml'}
    missing = [name for name, module in required_modules.items() if importlib.util.find_spec(module) is None]
    if missing:
        raise ModuleNotFoundError('Dependency kernel belum lengkap: ' + ', '.join(missing) + f'. Jalankan {sys.executable} -m pip install -r requirements.txt')
    print('Dependency lokal terverifikasi pada kernel aktif.')

Dependency lokal terverifikasi pada kernel aktif.


In [4]:
# Dataset Path dan Checksum
import hashlib
import warnings

# Opsi Google Drive (aktifkan dan sesuaikan hanya bila diperlukan):
# from google.colab import drive
# drive.mount('/content/drive')
# os.environ['SENSOR_DATA_PATH'] = '/content/drive/MyDrive/.../sensor_data.csv'

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

configured_path = os.environ.get('SENSOR_DATA_PATH', '').strip()
default_path = (REPO_ROOT / 'data/raw/sensor_data.csv').resolve()
if configured_path:
    DATASET_PATH = Path(configured_path).expanduser()
    if not DATASET_PATH.is_absolute():
        DATASET_PATH = (REPO_ROOT / DATASET_PATH).resolve()
    path_source = 'environment variable SENSOR_DATA_PATH'
elif default_path.is_file():
    DATASET_PATH, path_source = default_path, 'data/raw/sensor_data.csv'
elif not IS_COLAB:
    candidates_by_file = {}
    for pattern in ('*/Data/sensor_data.csv', '*/data/sensor_data.csv'):
        for path in REPO_ROOT.parent.glob(pattern):
            if path.is_file():
                info = path.stat()
                candidates_by_file[(info.st_dev, info.st_ino)] = path.resolve()
    candidates = sorted(candidates_by_file.values())
    if len(candidates) > 1:
        raise RuntimeError('Lebih dari satu dataset ditemukan; tetapkan SENSOR_DATA_PATH secara eksplisit.')
    DATASET_PATH = candidates[0] if candidates else default_path
    path_source = 'satu kandidat pada folder proyek saudara' if candidates else 'data/raw/sensor_data.csv'
else:
    DATASET_PATH, path_source = default_path, 'data/raw/sensor_data.csv'

if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'sensor_data.csv tidak ditemukan di {DATASET_PATH}. Tetapkan SENSOR_DATA_PATH atau gunakan Google Drive.')
DATASET_SHA256 = sha256_file(DATASET_PATH)
CHECKSUM_MATCH = DATASET_SHA256 == EXPECTED_DATASET_SHA256
print('Dataset path:', DATASET_PATH)
print('Sumber resolusi path:', path_source)
print('SHA-256:', DATASET_SHA256)
print('Checksum sesuai dataset penelitian:', CHECKSUM_MATCH)
if not CHECKSUM_MATCH:
    warnings.warn('Checksum berbeda; evaluasi tidak dijalankan tanpa review eksplisit.', RuntimeWarning)
    raise ValueError('Dataset berbeda dari dataset penelitian.')

Dataset path: /Users/macbookpro/Documents/jurnal_penelitian/data/sensor_data.csv
Sumber resolusi path: satu kandidat pada folder proyek saudara
SHA-256: ca7831a188a191edbf82a673fac90dbb875b5095986ed07699c02530f2a02a0e
Checksum sesuai dataset penelitian: True


In [5]:
# Load Canonical Configuration
import pandas as pd
import yaml
from IPython.display import display

with Path('configs/experiment.yaml').open(encoding='utf-8') as handle:
    experiment_config = yaml.safe_load(handle)
canonical_config = experiment_config['canonical_state']
display(pd.DataFrame([
    {'setting': 'schema_version', 'value': canonical_config['schema_version']},
    {'setting': 'room_id', 'value': canonical_config['room_id']},
    {'setting': 'source_timezone', 'value': experiment_config['data']['source_timezone']},
    {'setting': 'timezone_policy', 'value': experiment_config['preprocessing']['timezone_policy']},
    {'setting': 'gap_threshold_seconds', 'value': experiment_config['preprocessing']['gap_threshold_seconds']},
    {'setting': 'staleness_seconds', 'value': canonical_config['staleness_seconds']},
]))

,setting,value
0,schema_version,1.0.0-research
1,room_id,unresolved
2,source_timezone,UTC
3,timezone_policy,localize_naive_clock_value_as_utc
4,gap_threshold_seconds,60
5,staleness_seconds,None


In [6]:
# Raw Dataset Summary
dataset_summary = pd.read_csv('results/tables/dataset_summary.csv')
summary_sha = str(dataset_summary.loc[dataset_summary['metrik'] == 'sha256', 'nilai'].iloc[0])
if summary_sha != DATASET_SHA256:
    raise AssertionError('Dataset summary Tahap 2 tidak cocok dengan dataset aktif.')
display(dataset_summary)

,metrik,nilai,satuan,keterangan
0,nama_file,sensor_data.csv,NaN,Nama file sumber; path lokal tidak dipublikasikan
1,sha256,ca7831a188a191edbf82a673fac90dbb875b5095986ed0...,NaN,Checksum file sumber
2,jumlah_record,2027520,baris,Tidak termasuk header
3,jumlah_kolom,8,kolom,Schema CSV yang divalidasi
4,waktu_awal_utc,2026-02-23T23:14:43.896301Z,UTC,Minimum timestamp setelah interpretasi UTC
5,waktu_akhir_utc,2026-05-24T01:22:06.727676Z,UTC,Maksimum timestamp setelah interpretasi UTC
6,durasi_observasi_detik,7697242.831375,detik,Waktu akhir dikurangi waktu awal
7,durasi_observasi_hari,89.0884586964699,hari,Durasi detik dibagi 86.400
8,jumlah_device,1,device,DeviceID unik non-kosong
9,jumlah_timestamp_valid,2027520,baris,Timestamp yang dapat diparse


In [7]:
# Canonical Transformation Evaluation
import json
from src.twin_state.evaluation import evaluate_canonical_state

MANIFEST_PATH = Path('results/metrics/canonical_state_evaluation_manifest.json')
FORCE_RERUN = os.environ.get('FORCE_CANONICAL_EVALUATION', '0') == '1'
manifest_current = False
if MANIFEST_PATH.is_file():
    existing = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    code_current = all(Path(path).is_file() and sha256_file(path) == checksum for path, checksum in existing.get('code_sha256', {}).items())
    manifest_current = existing.get('input_sha256') == DATASET_SHA256 and existing.get('config_sha256') == sha256_file('configs/experiment.yaml') and code_current
if FORCE_RERUN or not manifest_current:
    evaluation_result = evaluate_canonical_state(input_path=DATASET_PATH)
    print('Pipeline evaluasi canonical dijalankan ulang.')
else:
    print('Output resmi yang cocok dengan dataset, konfigurasi, dan source code dimuat ulang.')
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
display(pd.DataFrame([manifest['summary']]).T.rename(columns={0: 'value'}))

Output resmi yang cocok dengan dataset, konfigurasi, dan source code dimuat ulang.


,value
total_raw_records_evaluated,2027520.0
successful_canonical_transformations,2027520.0
failed_canonical_transformations,0.0
schema_conforming_records,2027520.0
schema_conformity_rate,1.0
required_field_completeness_rate,1.0
data_type_validity_rate,1.0
timestamp_validity_rate,1.0
value_preservation_rate,1.0
determinism_checks,204.0


In [8]:
# Schema Conformity
evaluation = pd.read_csv('results/tables/canonical_state_evaluation.csv')
display(evaluation[evaluation['category'].isin(['transformation', 'schema', 'type', 'timestamp'])])

,category,metric,value,denominator,rate,unit,status,definition
0,transformation,total_raw_records_evaluated,2027520.0,NaN,NaN,record,observed,Seluruh baris data selain header
1,transformation,successful_canonical_transformations,2027520.0,2027520.0,1.0,record,pass,Transformer mengembalikan canonical state tanp...
2,transformation,failed_canonical_transformations,0.0,2027520.0,0.0,record,pass,Transformer menimbulkan exception
3,schema,schema_conforming_records,2027520.0,2027520.0,1.0,record,pass,"Struktur, field wajib, tipe nullable, dan meta..."
4,schema,schema_conformity_rate,1.0,NaN,NaN,ratio,pass,schema_conforming_records / total_raw_records_...
7,type,data_type_valid_records,2027520.0,2027520.0,1.0,record,pass,"Tidak memiliki invalid numeric, non-finite, at..."
8,timestamp,timestamp_valid_records,2027520.0,2027520.0,1.0,record,pass,Timestamp dapat diparse dan direpresentasikan ...
9,timestamp,timestamp_naive_localized_as_utc,2027520.0,2027520.0,1.0,record,observed,Clock value naive diinterpretasikan sebagai UT...
10,timestamp,timestamp_aware_converted_to_utc,0.0,2027520.0,0.0,record,observed,Timestamp dengan offset dikonversi ke UTC


In [9]:
# Source-to-State Mapping
field_mapping = pd.read_csv('results/tables/canonical_field_mapping.csv')
display(field_mapping)
if not (field_mapping['mapping_status'] == 'pass').all():
    raise AssertionError('Ditemukan mapping source-to-state yang tidak mempertahankan nilai.')

,source_field,canonical_field,source_type,canonical_type,unit,transformation,validation_rule,value_preservation,evaluated_records,non_null_canonical_values,completeness_rate,value_mismatches,mapping_status
0,Timestamp,timestamp_utc,string,ISO 8601 UTC string,UTC,Parse ISO 8601; timestamp tanpa suffix dilokal...,"Wajib tersedia, dapat diparse, dan hasil canon...",Instant dan presisi mikrodetik dipertahankan; ...,2027520,2027520,1.0,0,pass
1,DeviceID,device_id,string,string,identifier,Trim whitespace; tanpa pemetaan identitas baru,Wajib tersedia dan non-kosong,Isi identifier dipertahankan setelah trim whit...,2027520,2027520,1.0,0,pass
2,Suhu (C),environment.temperature_c,numeric string,float,°C,Parse float tanpa scaling atau konversi unit,Finite dan berada pada range konfigurasi,Nilai numerik dipertahankan,2027520,2027520,1.0,0,pass
3,Kelembaban (%),environment.humidity_percent,numeric string,float,%,Parse float tanpa scaling atau konversi unit,Finite dan berada pada range konfigurasi,Nilai numerik dipertahankan,2027520,2027520,1.0,0,pass
4,Tegangan (V),electrical.voltage_v,numeric string,float,V,Parse float tanpa scaling atau konversi unit,Finite dan berada pada range konfigurasi,Nilai numerik dipertahankan,2027520,2027520,1.0,0,pass
5,Arus (A),electrical.current_a,numeric string,float,A,Parse float tanpa scaling atau konversi unit,Finite dan berada pada range konfigurasi,Nilai numerik dipertahankan,2027520,2027520,1.0,0,pass
6,Daya (W),electrical.power_w,numeric string,float,W,Parse float tanpa scaling atau konversi unit,Finite dan berada pada range konfigurasi,Nilai numerik dipertahankan,2027520,2027520,1.0,0,pass
7,Jumlah Orang,occupancy.count,numeric string,integer,orang,Parse numeric lalu validasi integer; tanpa sca...,Integer finite dan berada pada range konfigurasi,Nilai hitungan integer dipertahankan,2027520,2027520,1.0,0,pass


In [10]:
# State Completeness
data_quality = pd.read_csv('results/tables/digital_twin_data_quality.csv')
display(data_quality[data_quality['category'].isin(['expected_from_telemetry', 'metadata', 'derived_quality'])])
print('room_id dan staleness_seconds tidak menurunkan completeness telemetry karena keduanya bukan field telemetry yang tersedia pada CSV.')

,category,item,count,denominator,rate,status,interpretation
0,expected_from_telemetry,timestamp_utc,2027520,2027520.0,1.0,complete,Masuk denominator required-field completeness
1,expected_from_telemetry,device_id,2027520,2027520.0,1.0,complete,Masuk denominator required-field completeness
2,expected_from_telemetry,environment.temperature_c,2027520,2027520.0,1.0,complete,Masuk denominator required-field completeness
3,expected_from_telemetry,environment.humidity_percent,2027520,2027520.0,1.0,complete,Masuk denominator required-field completeness
4,expected_from_telemetry,electrical.voltage_v,2027520,2027520.0,1.0,complete,Masuk denominator required-field completeness
5,expected_from_telemetry,electrical.current_a,2027520,2027520.0,1.0,complete,Masuk denominator required-field completeness
6,expected_from_telemetry,electrical.power_w,2027520,2027520.0,1.0,complete,Masuk denominator required-field completeness
7,expected_from_telemetry,occupancy.count,2027520,2027520.0,1.0,complete,Masuk denominator required-field completeness
8,metadata,room_id_resolved,0,2027520.0,0.0,unresolved_by_source,Tidak masuk denominator telemetry; CSV tidak m...
9,derived_quality,data_quality.valid_available,2027520,2027520.0,1.0,available,"Boolean diturunkan dari rule missing, timestam..."


room_id dan staleness_seconds tidak menurunkan completeness telemetry karena keduanya bukan field telemetry yang tersedia pada CSV.


In [11]:
# Temporal Integrity
display(evaluation[evaluation['category'].isin(['timestamp', 'temporal'])])
print('UTC dinormalisasi berdasarkan provenance; tidak ada klaim physical-to-digital latency.')

,category,metric,value,denominator,rate,unit,status,definition
8,timestamp,timestamp_valid_records,2027520.0,2027520.0,1.0,record,pass,Timestamp dapat diparse dan direpresentasikan ...
9,timestamp,timestamp_naive_localized_as_utc,2027520.0,2027520.0,1.0,record,observed,Clock value naive diinterpretasikan sebagai UT...
10,timestamp,timestamp_aware_converted_to_utc,0.0,2027520.0,0.0,record,observed,Timestamp dengan offset dikonversi ke UTC
11,temporal,non_monotonic_source_pairs,0.0,NaN,NaN,record,pass,Pasangan timestamp berurutan dalam file dengan...
12,temporal,duplicate_timestamp_rows,0.0,NaN,NaN,record,pass,Baris berlebih dengan timestamp UTC yang sama
13,temporal,duplicate_timestamp_groups,0.0,NaN,NaN,record,pass,Kelompok timestamp UTC dengan lebih dari satu ...
14,temporal,temporal_gaps,176.0,NaN,NaN,record,observed,Interval setelah pengurutan yang lebih besar d...
15,temporal,deterministic_ordering_verified,1.0,NaN,NaN,boolean,pass,Urutan canonical ditentukan oleh timestamp_utc...


UTC dinormalisasi berdasarkan provenance; tidak ada klaim physical-to-digital latency.


In [12]:
# Data-Quality Evaluation
display(data_quality[data_quality['category'].isin(['validation_behavior', 'limitation'])])
print('Synthetic invalid cases diuji melalui unit test dan tidak dicampurkan dengan hasil dataset aktual.')

,category,item,count,denominator,rate,status,interpretation
11,validation_behavior,records_with_missing_required_field,0,2027520.0,0.0,pass,Missing dipertahankan sebagai null dan menghas...
12,validation_behavior,records_with_invalid_type,0,2027520.0,0.0,pass,"Invalid numeric, non-finite, dan non-integer m..."
13,validation_behavior,records_with_invalid_timestamp,0,2027520.0,0.0,pass,Timestamp invalid menjadi null dan valid=false
14,validation_behavior,records_with_range_violation,0,2027520.0,0.0,pass,Nilai dipertahankan tetapi valid=false sesuai ...
15,limitation,occupancy_staleness_calculable,0,NaN,NaN,not_evaluable,Tidak ada timestamp kamera independen; stalene...


Synthetic invalid cases diuji melalui unit test dan tidak dicampurkan dengan hasil dataset aktual.


In [13]:
# Transformation Correctness
correctness = evaluation[evaluation['category'] == 'correctness']
display(correctness)
display(field_mapping[['source_field', 'canonical_field', 'evaluated_records', 'value_mismatches', 'mapping_status']])

,category,metric,value,denominator,rate,unit,status,definition
17,correctness,value_preservation_comparisons,16220160.0,NaN,NaN,record,observed,Perbandingan source-to-state pada nilai sumber...
18,correctness,value_preservation_mismatches,0.0,16220160.0,0.0,record,pass,Nilai canonical berbeda dari nilai sumber sete...
19,correctness,value_preservation_rate,1.0,NaN,NaN,ratio,pass,Perbandingan source-to-state yang mempertahank...


,source_field,canonical_field,evaluated_records,value_mismatches,mapping_status
0,Timestamp,timestamp_utc,2027520,0,pass
1,DeviceID,device_id,2027520,0,pass
2,Suhu (C),environment.temperature_c,2027520,0,pass
3,Kelembaban (%),environment.humidity_percent,2027520,0,pass
4,Tegangan (V),electrical.voltage_v,2027520,0,pass
5,Arus (A),electrical.current_a,2027520,0,pass
6,Daya (W),electrical.power_w,2027520,0,pass
7,Jumlah Orang,occupancy.count,2027520,0,pass


In [14]:
# Determinism Checks
determinism = evaluation[evaluation['category'] == 'determinism']
display(determinism)
if int(determinism.loc[determinism['metric'] == 'determinism_failures', 'value'].iloc[0]) != 0:
    raise AssertionError('Transformasi canonical tidak deterministik.')

,category,metric,value,denominator,rate,unit,status,definition
20,determinism,records_rechecked,204.0,NaN,NaN,record,observed,"Record pertama, terakhir, dan setiap 10,000 re..."
21,determinism,determinism_failures,0.0,204.0,0.0,record,pass,Transformasi ulang raw record yang sama mengha...


In [15]:
# Results Tables
print('Tabel resmi:')
for name, path in manifest['official_outputs'].items():
    output_path = Path(path)
    print(f'- {name}: {output_path} ({"tersedia" if output_path.is_file() else "tidak tersedia"})')

Tabel resmi:
- canonical_state_evaluation: results/tables/canonical_state_evaluation.csv (tersedia)
- canonical_field_mapping: results/tables/canonical_field_mapping.csv (tersedia)
- digital_twin_data_quality: results/tables/digital_twin_data_quality.csv (tersedia)


In [16]:
# Limitations
from IPython.display import Markdown
limitations = '\n'.join(f'- {item}' for item in manifest['limitations'])
display(Markdown('### Keterbatasan yang tidak dapat dievaluasi\n\n' + limitations))

### Keterbatasan yang tidak dapat dievaluasi

- Tidak tersedia timestamp independen camera capture.
- Tidak tersedia timestamp independen sensor measurement.
- Tidak tersedia timestamp independen gateway arrival.
- Tidak tersedia timestamp independen cloud ingestion.
- Physical-to-digital latency, occupancy staleness, exact camera-sensor synchronization, dan end-to-end synchronization performance tidak dapat dihitung.

In [17]:
# Reproducibility Summary
reproducibility = {
    'dataset_sha256': manifest['input_sha256'],
    'config_sha256': manifest['config_sha256'],
    'code_sha256': manifest['code_sha256'],
    'git_commit': manifest.get('git_commit'),
    'working_tree_dirty': manifest.get('working_tree_dirty'),
    'python_version': sys.version.split()[0],
    'schema_version': manifest['schema_version'],
    'generated_at_utc': manifest['generated_at_utc'],
    'official_outputs': manifest['official_outputs'],
}
print(json.dumps(reproducibility, ensure_ascii=False, indent=2))

{
  "dataset_sha256": "ca7831a188a191edbf82a673fac90dbb875b5095986ed07699c02530f2a02a0e",
  "config_sha256": "39af52c562fe3cdd671107b87acb91adc9242a640c3c64f81e2466086c774d5c",
  "code_sha256": {
    "src/twin_state/evaluation.py": "3075cbd1f79fe056afb0ba8d019148df5d747b097acac0431582853b40b78dd3",
    "src/twin_state/canonical.py": "403e16a1bb4a4e51398fc6a34543700bc46969871d06ea6b70b9be142d2b8694"
  },
  "git_commit": "cd71c0b8b14b0cc4aef4fda5af5bff5ced964480",
  "working_tree_dirty": true,
  "python_version": "3.11.9",
  "schema_version": "1.0.0-research",
  "generated_at_utc": "2026-09-24T11:55:43.082217+00:00",
  "official_outputs": {
    "canonical_state_evaluation": "results/tables/canonical_state_evaluation.csv",
    "canonical_field_mapping": "results/tables/canonical_field_mapping.csv",
    "digital_twin_data_quality": "results/tables/digital_twin_data_quality.csv"
  }
}
